# DASC8412 Assignment 2: Question 7
## Generative AI code comparison: two models, one prompt

This notebook runs the two scripts produced by the two generative AI models and saves their plots. It contains no other analysis. The scripts are pasted exactly as the models produced them.

| | Model 1 | Model 2 |
|---|---|---|
| Tool | ChatGPT | Gemini |
| Version | GPT-6 | 3.5 Flash-Lite |
| Chat link | https://chatgpt.com/share/6abf5225-7f20-83ea-8fd1-c5acdbed471a | https://share.gemini.google/1xayJkzT0Op8 |

**To run:** place `School_Data_CSV.csv` in a folder called `data` next to this notebook, then choose Run All. Each model's PNG files are saved in an `outputs` folder with `_model1` or `_model2` added to the file name.

### The prompt (identical for both models)

```text
You are a senior Python data analyst. You write clear, well-commented scripts that run the first time and that cope with messy real-world files.

Context: I have a CSV file called data/School_Data_CSV.csv about South African schools, with about 200,000 rows. Its nine columns are: emisno, centername, institution_name, centreno, quintile, ward_id, street_address, new_long, new_lat. I work in Python 3.10 with pandas and matplotlib.

Notes about the data, found by checking the file:
- The file is not UTF-8. It uses a Windows encoding.
- Most schools appear in several rows (typically about six). emisno identifies a school, but the value 99 is a placeholder, not a real school. There are also exact duplicate rows.
- quintile is written in several formats: 1 to 5, Q1 to Q5 and NQ1 to NQ5. Some rows hold placeholder text such as "TO BE UPDATED", "NOT APPLICABLE", "UNKNOWN" or ".", or the number 99. Only the values 1 to 5 are valid quintiles.
- new_long and new_lat contain text such as TRUE and "." as well as numbers. A large share of new_lat is missing or not numeric. Some numeric values are impossible. Valid South African coordinates fall roughly between latitude -35 and -22 and between longitude 16 and 33.
- ward_id mixes formats. Most values are 8-digit ward codes, but some are short numbers of 1 to 3 digits, some are longer, and some are placeholders. There are more than 4,000 distinct ward_id values.

Task: write one complete, runnable Python script that reads the file and produces three plots, counting schools rather than rows.
(a) A line plot showing how the number of schools per quintile changes across wards. Because there are thousands of wards, choose an approach that keeps the plot readable and state your choice in a code comment.
(b) A scatter plot showing the geographic locations of schools using new_long and new_lat.
(c) A histogram showing the distribution of schools by quintile.

Constraints:
- Use only the Python standard library, pandas and matplotlib.
- Give every plot a title and labelled axes, and add a legend wherever a plot shows more than one series.
- Save each plot as its own PNG file. Do not rely on plt.show().
- Print a short summary of how many rows or schools were excluded from each plot and why.
- Deliver everything as one script that I can run with: python script.py

Output format: one Python code block containing the whole script, followed by no more than five sentences explaining your main design choices.
```

### Model 1: ChatGPT (GPT-6)

In [1]:
# MODEL 1: ChatGPT (GPT-6). Script exactly as generated.
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

INPUT_FILE = Path("data/School_Data_CSV.csv")
OUTPUT_DIR = Path("plots")

# The file is known to use a Windows encoding rather than UTF-8.
# cp1252 is the usual Windows Western European encoding used by CSV exports.
ENCODING = "cp1252"

VALID_QUINTILES = {1, 2, 3, 4, 5}

# Approximate geographic bounds for South Africa.
MIN_LAT, MAX_LAT = -35.0, -22.0
MIN_LONG, MAX_LONG = 16.0, 33.0

# A normal ward code in this dataset is expected to contain exactly 8 digits.
WARD_PATTERN = re.compile(r"^\d{8}$")

REQUIRED_COLUMNS = [
    "emisno",
    "centername",
    "institution_name",
    "centreno",
    "quintile",
    "ward_id",
    "street_address",
    "new_long",
    "new_lat",
]


# ---------------------------------------------------------------------------
# Cleaning functions
# ---------------------------------------------------------------------------

def clean_quintile(value):
    """
    Convert valid quintile representations such as:
        1, 2, ..., 5
        Q1, Q2, ..., Q5
        NQ1, NQ2, ..., NQ5

    to integers 1-5.

    Invalid values such as 99, UNKNOWN, TO BE UPDATED, NOT APPLICABLE and '.'
    become pandas.NA.
    """
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().upper()

    # Remove optional Q or NQ prefixes.
    match = re.fullmatch(r"(?:NQ|Q)?([1-5])(?:\.0)?", text)

    if match:
        return int(match.group(1))

    return pd.NA


def clean_ward(value):
    """
    Keep only normal 8-digit ward codes.

    Short 1-3 digit values, unusually long values, placeholders and other
    malformed ward IDs are excluded because mixing them with normal ward
    codes could produce misleading ward-level comparisons.
    """
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    # Handle cases where a numeric ward was stored as e.g. 12345678.0.
    if text.endswith(".0"):
        text = text[:-2]

    if WARD_PATTERN.fullmatch(text):
        return text

    return pd.NA


def normalise_emisno(value):
    """
    Normalise EMIS numbers for school-level deduplication.

    The placeholder EMIS number 99 is treated as missing because it does not
    uniquely identify a school.
    """
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    if text.endswith(".0"):
        text = text[:-2]

    if text == "" or text.upper() in {
        "99",
        ".",
        "UNKNOWN",
        "NOT APPLICABLE",
        "TO BE UPDATED",
    }:
        return pd.NA

    return text


def first_valid(series):
    """
    Return the first non-missing value in a Series.

    Used when several source rows represent the same school.
    """
    valid = series.dropna()

    if valid.empty:
        return pd.NA

    return valid.iloc[0]


# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------

def load_data():
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"Could not find '{INPUT_FILE}'.\n"
            "Run the script from the project directory containing the data folder."
        )

    print(f"Reading {INPUT_FILE}...")

    try:
        df = pd.read_csv(
            INPUT_FILE,
            encoding=ENCODING,
            dtype=str,
            low_memory=False,
        )
    except UnicodeDecodeError:
        # latin-1 can decode every byte and is a useful fallback for messy
        # Windows-generated files if cp1252 unexpectedly fails.
        print(
            "Warning: cp1252 decoding failed. "
            "Retrying with latin-1 encoding."
        )
        df = pd.read_csv(
            INPUT_FILE,
            encoding="latin-1",
            dtype=str,
            low_memory=False,
        )

    # Protect against accidental whitespace in CSV headers.
    df.columns = df.columns.str.strip().str.lower()

    missing_columns = [
        column for column in REQUIRED_COLUMNS if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            "The CSV is missing required columns: "
            + ", ".join(missing_columns)
        )

    # Keep only columns needed by this analysis.
    df = df[REQUIRED_COLUMNS].copy()

    return df


# ---------------------------------------------------------------------------
# Prepare one record per school
# ---------------------------------------------------------------------------

def prepare_school_data(df):
    original_rows = len(df)

    # Remove completely identical source records before any further processing.
    df = df.drop_duplicates().copy()
    exact_duplicates_removed = original_rows - len(df)

    # Clean the fields before school-level aggregation.
    df["emisno_clean"] = df["emisno"].map(normalise_emisno)
    df["quintile_clean"] = df["quintile"].map(clean_quintile).astype("Int64")
    df["ward_clean"] = df["ward_id"].map(clean_ward)

    # Invalid coordinate strings such as TRUE and "." become NaN.
    df["longitude_clean"] = pd.to_numeric(df["new_long"], errors="coerce")
    df["latitude_clean"] = pd.to_numeric(df["new_lat"], errors="coerce")

    placeholder_emis_rows = df["emisno_clean"].isna().sum()

    # Counting schools requires a genuine school identifier. EMIS 99 and other
    # missing/placeholder identifiers cannot safely be deduplicated because
    # many unrelated schools could otherwise be collapsed into one school.
    identifiable = df[df["emisno_clean"].notna()].copy()

    # A school may occur in several rows. Collapse those rows to one record
    # using EMIS number, taking the first usable value for each plot variable.
    schools = (
        identifiable.groupby("emisno_clean", as_index=False)
        .agg(
            quintile=("quintile_clean", first_valid),
            ward_id=("ward_clean", first_valid),
            longitude=("longitude_clean", first_valid),
            latitude=("latitude_clean", first_valid),
        )
    )

    schools["quintile"] = schools["quintile"].astype("Int64")

    print("\nDATA CLEANING SUMMARY")
    print("-" * 60)
    print(f"Rows read:                         {original_rows:,}")
    print(f"Exact duplicate rows removed:     {exact_duplicates_removed:,}")
    print(
        f"Rows without a usable EMIS number: {placeholder_emis_rows:,} "
        "(includes EMIS 99)"
    )
    print(f"Unique identifiable schools:      {len(schools):,}")

    return schools


# ---------------------------------------------------------------------------
# Plot A: Schools per quintile across wards
# ---------------------------------------------------------------------------

def plot_schools_by_ward(schools):
    valid = schools[
        schools["ward_id"].notna()
        & schools["quintile"].isin(VALID_QUINTILES)
    ].copy()

    missing_ward = schools["ward_id"].isna().sum()
    invalid_quintile = schools["quintile"].isna().sum()

    # There are thousands of wards, so plotting every ward would produce an
    # unreadable chart. Instead, select the 30 wards with the largest number
    # of schools that also have a valid quintile. The same 30 wards are used
    # for all five quintile series, making comparisons readable and consistent.
    ward_totals = (
        valid.groupby("ward_id")
        .size()
        .sort_values(ascending=False)
    )

    top_wards = ward_totals.head(30).index

    plot_data = (
        valid[valid["ward_id"].isin(top_wards)]
        .groupby(["ward_id", "quintile"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=range(1, 6), fill_value=0)
    )

    # Sort selected wards by their total number of schools so the line plot
    # has a meaningful and reproducible left-to-right ordering.
    plot_data["total"] = plot_data.sum(axis=1)
    plot_data = plot_data.sort_values("total", ascending=False)
    plot_data = plot_data.drop(columns="total")

    fig, ax = plt.subplots(figsize=(16, 8))

    for quintile in range(1, 6):
        ax.plot(
            range(len(plot_data)),
            plot_data[quintile],
            marker="o",
            markersize=4,
            linewidth=1.5,
            label=f"Quintile {quintile}",
        )

    ax.set_title("Number of Schools per Quintile Across the 30 Largest Wards")
    ax.set_xlabel("Ward ID")
    ax.set_ylabel("Number of Schools")
    ax.set_xticks(range(len(plot_data)))
    ax.set_xticklabels(plot_data.index, rotation=90)
    ax.legend(title="Quintile")
    ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "schools_per_quintile_by_ward.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

    valid_not_top30 = len(valid) - len(
        valid[valid["ward_id"].isin(top_wards)]
    )

    print("\nPLOT A: SCHOOLS PER QUINTILE ACROSS WARDS")
    print("-" * 60)
    print(f"Schools available after validation: {len(valid):,}")
    print(
        f"Schools excluded for invalid/missing ward ID: {missing_ward:,}"
    )
    print(
        f"Schools with invalid/missing quintile:         {invalid_quintile:,}"
    )
    print(
        f"Valid schools outside the top 30 wards:        "
        f"{valid_not_top30:,}"
    )
    print(
        "Note: exclusion counts can overlap because a school may have "
        "more than one invalid field."
    )


# ---------------------------------------------------------------------------
# Plot B: Geographic locations
# ---------------------------------------------------------------------------

def plot_school_locations(schools):
    numeric_coordinates = (
        schools["longitude"].notna()
        & schools["latitude"].notna()
    )

    within_bounds = (
        schools["longitude"].between(MIN_LONG, MAX_LONG)
        & schools["latitude"].between(MIN_LAT, MAX_LAT)
    )

    valid_mask = numeric_coordinates & within_bounds
    valid = schools[valid_mask].copy()

    missing_or_non_numeric = (~numeric_coordinates).sum()
    impossible_coordinates = (
        numeric_coordinates & ~within_bounds
    ).sum()

    fig, ax = plt.subplots(figsize=(10, 10))

    ax.scatter(
        valid["longitude"],
        valid["latitude"],
        s=8,
        alpha=0.35,
    )

    ax.set_title("Geographic Locations of South African Schools")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_xlim(MIN_LONG, MAX_LONG)
    ax.set_ylim(MIN_LAT, MAX_LAT)
    ax.grid(alpha=0.25)

    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "school_geographic_locations.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

    print("\nPLOT B: GEOGRAPHIC SCHOOL LOCATIONS")
    print("-" * 60)
    print(f"Schools plotted:                         {len(valid):,}")
    print(
        f"Excluded: missing/non-numeric coordinate: "
        f"{missing_or_non_numeric:,}"
    )
    print(
        f"Excluded: coordinates outside SA bounds:  "
        f"{impossible_coordinates:,}"
    )


# ---------------------------------------------------------------------------
# Plot C: Quintile distribution
# ---------------------------------------------------------------------------

def plot_quintile_histogram(schools):
    valid = schools[
        schools["quintile"].isin(VALID_QUINTILES)
    ].copy()

    invalid_quintile = len(schools) - len(valid)

    fig, ax = plt.subplots(figsize=(10, 6))

    # Half-integer bin edges ensure integer quintiles 1-5 each occupy one bin.
    bins = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]

    ax.hist(
        valid["quintile"].astype(int),
        bins=bins,
        edgecolor="black",
    )

    ax.set_title("Distribution of Schools by Quintile")
    ax.set_xlabel("Quintile")
    ax.set_ylabel("Number of Schools")
    ax.set_xticks(range(1, 6))
    ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "school_quintile_distribution.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

    print("\nPLOT C: SCHOOL QUINTILE DISTRIBUTION")
    print("-" * 60)
    print(f"Schools included:                    {len(valid):,}")
    print(
        f"Schools excluded: invalid/missing quintile: "
        f"{invalid_quintile:,}"
    )


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = load_data()
    schools = prepare_school_data(df)

    if schools.empty:
        raise ValueError(
            "No schools with usable EMIS numbers remain after cleaning."
        )

    plot_schools_by_ward(schools)
    plot_school_locations(schools)
    plot_quintile_histogram(schools)

    print("\nOUTPUT FILES")
    print("-" * 60)
    print(OUTPUT_DIR / "schools_per_quintile_by_ward.png")
    print(OUTPUT_DIR / "school_geographic_locations.png")
    print(OUTPUT_DIR / "school_quintile_distribution.png")
    print("\nAnalysis completed successfully.")


if __name__ == "__main__":
    main()

# ---------------------------------------------------------------
# Added by me (not part of the model's script): collect this
# model's PNG files into outputs/ and tag each name with the model.
# ---------------------------------------------------------------
import shutil
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)
for png in Path("plots").glob("*.png"):
    target = Path("outputs") / (png.stem + "_model1.png")
    shutil.move(str(png), str(target))
    print("Saved:", target)


Reading data/School_Data_CSV.csv...



DATA CLEANING SUMMARY
------------------------------------------------------------
Rows read:                         199,853
Exact duplicate rows removed:     799
Rows without a usable EMIS number: 1,550 (includes EMIS 99)
Unique identifiable schools:      33,669



PLOT A: SCHOOLS PER QUINTILE ACROSS WARDS
------------------------------------------------------------
Schools available after validation: 24,730
Schools excluded for invalid/missing ward ID: 6,014
Schools with invalid/missing quintile:         8,480
Valid schools outside the top 30 wards:        24,018
Note: exclusion counts can overlap because a school may have more than one invalid field.



PLOT B: GEOGRAPHIC SCHOOL LOCATIONS
------------------------------------------------------------
Schools plotted:                         27,421
Excluded: missing/non-numeric coordinate: 5,936
Excluded: coordinates outside SA bounds:  312



PLOT C: SCHOOL QUINTILE DISTRIBUTION
------------------------------------------------------------
Schools included:                    25,189
Schools excluded: invalid/missing quintile: 8,480

OUTPUT FILES
------------------------------------------------------------
plots/schools_per_quintile_by_ward.png
plots/school_geographic_locations.png
plots/school_quintile_distribution.png

Analysis completed successfully.
Saved: outputs/school_geographic_locations_model1.png
Saved: outputs/schools_per_quintile_by_ward_model1.png
Saved: outputs/school_quintile_distribution_model1.png


### Model 2: Gemini (3.5 Flash-Lite)

In [2]:
# MODEL 2: Gemini (3.5 Flash-Lite). Script exactly as generated.
import re
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. LOAD DATA & HANDLE ENCODING
# ---------------------------------------------------------
# The file uses a Windows encoding (commonly cp1252 or latin1 for legacy SA datasets)
file_path = "data/School_Data_CSV.csv"

try:
    df = pd.read_csv(file_path, encoding="cp1252", low_memory=False)
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding="latin1", low_memory=False)

initial_row_count = len(df)
exclusion_log = {}

# ---------------------------------------------------------
# 2. DATA CLEANING & PREPROCESSING
# ---------------------------------------------------------
# Drop exact duplicate rows first
df = df.drop_duplicates()
exclusion_log['exact_duplicates'] = initial_row_count - len(df)

# Clean column names (strip whitespace just in case)
df.columns = df.columns.str.strip()

# Clean emisno: remove placeholder '99'
df['emisno'] = pd.to_numeric(df['emisno'], errors='coerce')
valid_emis = df['emisno'].notna() & (df['emisno'] != 99)
df_clean = df[valid_emis].copy()
exclusion_log['invalid_emisno'] = len(df) - len(df_clean)

# Standardize quintile: extract digits 1-5 from formats like 'Q1', 'NQ1', '1', etc.
def clean_quintile(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip().upper()
    if val_str in ['99', 'TO BE UPDATED', 'NOT APPLICABLE', 'UNKNOWN', '.']:
        return None
    # Search for a single digit between 1 and 5
    match = re.search(r'[1-5]', val_str)
    if match:
        return int(match.group(0))
    return None

df_clean['clean_quintile'] = df_clean['quintile'].apply(clean_quintile)

# Clean coordinates: filter for valid South African bounds
df_clean['new_long'] = pd.to_numeric(df_clean['new_long'], errors='coerce')
df_clean['new_lat'] = pd.to_numeric(df_clean['new_lat'], errors='coerce')

lat_valid = df_clean['new_lat'].between(-35.0, -22.0)
long_valid = df_clean['new_long'].between(16.0, 33.0)
df_clean['coords_valid'] = lat_valid & long_valid

# Unique schools per dataset (deduplicating by emisno for school-level counts)
# For ward analysis and quintile histogram, we want unique schools.
# Note: A school can change quintile or ward in dirty data, but we take the first/mode or drop if ambiguous.
# To keep counts clean, we drop rows with missing clean_quintile or invalid emisno.
df_schools = df_clean.dropna(subset=['clean_quintile', 'emisno']).drop_duplicates(subset=['emisno'])
exclusion_log['invalid_quintile_or_school'] = len(df_clean) - len(df_schools)

# ---------------------------------------------------------
# 3. PLOT (A): Line plot of schools per quintile across wards
# ---------------------------------------------------------
# Because there are thousands of wards, plotting thousands of individual lines is unreadable.
# Choice: Group wards into bins or aggregate data by sorting wards by total school count / 
# or plot top N wards, or display average distribution / cumulative distribution. 
# Here, we pivot to show the distribution across a sampled/aggregated subset of top wards 
# or sort wards by total schools and plot the top 50 densest wards to keep the line plot clean and legible.
ward_quintile = df_clean.dropna(subset=['clean_quintile', 'ward_id']).groupby(['ward_id', 'clean_quintile'])['emisno'].nunique().unstack(fill_value=0)
ward_quintile['total'] = ward_quintile.sum(axis=1)
# Select top 40 wards by school density for a readable line plot
top_wards = ward_quintile.sort_values(by='total', ascending=False).head(40).drop(columns=['total'])

plt.figure(figsize=(12, 6))
for q in sorted(top_wards.columns):
    plt.plot(top_wards.index.astype(str), top_wards[q], marker='o', label=f'Quintile {q}')

plt.title('Number of Schools per Quintile Across Top 40 Wards (Density Sorted)')
plt.xlabel('Ward ID')
plt.ylabel('Number of Schools')
plt.xticks(rotation=90, fontsize=8)
plt.legend(title='Quintile')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('schools_per_quintile_by_ward.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# 4. PLOT (B): Scatter plot of geographic locations
# ---------------------------------------------------------
geo_df = df_schools[df_schools['coords_valid']].copy()
exclusion_log['invalid_coordinates'] = len(df_schools) - len(geo_df)

plt.figure(figsize=(8, 10))
plt.scatter(geo_df['new_long'], geo_df['new_lat'], c=geo_df['clean_quintile'], cmap='viridis', s=2, alpha=0.5)
plt.colorbar(label='Quintile')
plt.title('Geographic Distribution of South African Schools')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('school_locations_scatter.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# 5. PLOT (C): Histogram of schools by quintile
# ---------------------------------------------------------
plt.figure(figsize=(8, 6))
quintile_counts = df_schools['clean_quintile'].value_counts().sort_index()
quintile_counts.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Distribution of Unique Schools by Quintile')
plt.xlabel('Quintile')
plt.ylabel('Number of Schools')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('schools_by_quintile_histogram.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# 6. SUMMARY REPORT
# ---------------------------------------------------------
print("=== EXCLUSION SUMMARY ===")
print(f"Initial row count: {initial_row_count}")
for reason, count in exclusion_log.items():
    print(f"- Excluded due to {reason}: {count} rows/records")
print(f"Final unique valid schools analyzed: {len(df_schools)}")
print("\nPlots successfully generated and saved as PNG files.")

# ---------------------------------------------------------------
# Added by me (not part of the model's script): collect this
# model's PNG files into outputs/ and tag each name with the model.
# ---------------------------------------------------------------
import shutil
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)
for png in Path(".").glob("*.png"):
    target = Path("outputs") / (png.stem + "_model2.png")
    shutil.move(str(png), str(target))
    print("Saved:", target)


=== EXCLUSION SUMMARY ===
Initial row count: 199853
- Excluded due to exact_duplicates: 799 rows/records
- Excluded due to invalid_emisno: 1568 rows/records
- Excluded due to invalid_quintile_or_school: 172297 rows/records
- Excluded due to invalid_coordinates: 22929 rows/records
Final unique valid schools analyzed: 25189

Plots successfully generated and saved as PNG files.
Saved: outputs/schools_per_quintile_by_ward_model2.png
Saved: outputs/school_locations_scatter_model2.png
Saved: outputs/schools_by_quintile_histogram_model2.png
